# 04 — Model Comparison with Cross-Validation

> **Workflow position:** Data is clean, features are engineered. Now we pit multiple algorithms against each other before committing to one.

---

## Why compare models before committing?

Aurélien Géron's *Hands-On Machine Learning* (2025), Chapter 1, reminds us of the **No Free Lunch theorem**: no single algorithm is universally best. A logistic regression might beat a neural network on a small, well-structured tabular dataset; a gradient boosting tree might outperform both on data with complex interactions.

The correct workflow is:
1. **Shortlist** several diverse model families (linear, tree-based ensembles, boosted trees).
2. **Cross-validate** each on the *same* folds to get an unbiased performance estimate.
3. **Select** the top 1-2 candidates for deeper hyperparameter tuning.
4. Only **commit** to one model after tuning — and never peek at the test set until the very end.

Skipping step 1-2 and going straight to tuning a model chosen by intuition is a common beginner mistake that wastes compute and leads to suboptimal results.

## Setup

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# --- project path setup ---
PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = sns.color_palette("Set2")
sns.set_palette(PALETTE)

print("Python path configured. Project root:", PROJECT_ROOT)

## Load Data & Apply Feature Engineering

We replicate the exact same preparation used for training so that the comparison is apples-to-apples.

In [ ]:
from client_churn_prediction.config import load_config
from client_churn_prediction.data import load_training_frame
from client_churn_prediction.features import model_matrix

config = load_config()
df_raw = load_training_frame(config)
X, y, df_featured = model_matrix(df_raw, config, training=True)

print(f"Dataset shape : {df_raw.shape}")
print(f"Feature matrix: {X.shape}")
print(f"Target         : {y.name} — classes {y.unique()} — churn rate {y.mean():.1%}")
X.head(3)

## Manual Baseline — Always-Predict-Majority-Class

Before running any real model, we establish a **dummy baseline**: a classifier that always predicts the majority class (no churn). Any real model that cannot beat this baseline is useless.

For imbalanced classification, ROC AUC is the preferred metric:
- **0.5** = random classifier (coin flip)
- **1.0** = perfect separation
- A dummy majority classifier scores exactly **0.5**

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# DummyClassifier predict_proba on majority-class returns all-same scores
# so ROC AUC is exactly 0.5 — confirmed below
dummy_auc = cross_val_score(dummy, X, y, cv=cv, scoring="roc_auc").mean()

churn_rate = y.mean()
print(f"Churn rate (positive class)  : {churn_rate:.1%}")
print(f"Majority-class baseline AUC  : {dummy_auc:.4f}")
print(f"Majority-class baseline acc  : {1 - churn_rate:.1%}")
print()
print("⚠  Accuracy is misleading here — predicting 'no churn' 100% of the time")
print(f"   gives {1 - churn_rate:.1%} accuracy but identifies ZERO churners.")
print("   We optimise for ROC AUC and Precision@k instead.")

## Cross-Validation Model Comparison

We use `compare_models()` from our `models` module, which:
- Builds a **ColumnTransformer** preprocessing pipeline (median imputation + StandardScaler for numerics; most-frequent imputation + OneHotEncoder for categoricals)
- Wraps each candidate estimator in a **sklearn Pipeline** to prevent data leakage
- Runs **StratifiedKFold(5)** cross-validation, preserving class ratios in every fold
- Returns ROC AUC, Average Precision, and F1 for each model

Models compared:
| Model | Strengths |
|---|---|
| Logistic Regression | Interpretable, fast, strong regularisation baseline |
| Random Forest | Handles non-linearity, robust to outliers, good out-of-the-box |
| Gradient Boosting | Often best on tabular data; iteratively corrects errors |
| Extra Trees | Fast training, high variance reduction via random splits |

In [ ]:
from client_churn_prediction.models import compare_models

print("Running 5-fold cross-validation for 4 model families...")
print("This may take 1-3 minutes on first run.\n")

results = compare_models()
print("Done!")

## Results Table — Sorted by ROC AUC

In [ ]:
results_df = pd.DataFrame(results).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)

# Pretty display
display_df = results_df.copy()
display_df.index += 1  # rank starts at 1
display_df.columns = [
    "Model", "ROC AUC (mean)", "ROC AUC (std)",
    "Avg Precision (mean)", "F1 (mean)"
]
display_df["Model"] = display_df["Model"].str.replace("_", " ").str.title()

print("=" * 70)
print("MODEL COMPARISON — 5-fold Stratified Cross-Validation")
print("=" * 70)
print(display_df.to_string())
print()
print(f"Baseline (dummy) ROC AUC: {dummy_auc:.4f}")

best_model_name = results_df.iloc[0]["model"]
best_auc = results_df.iloc[0]["roc_auc_mean"]
print(f"\nBest model  : {best_model_name.replace('_', ' ').title()}")
print(f"Best ROC AUC: {best_auc:.4f} (+{best_auc - dummy_auc:.4f} vs baseline)")

## Bar Chart — ROC AUC with Error Bars

Error bars show ±1 standard deviation across the 5 CV folds. A large error bar signals high sensitivity to the training sample — a potential overfitting risk.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

model_labels = results_df["model"].str.replace("_", "\n").str.title()
colors = [PALETTE[i % len(PALETTE)] for i in range(len(results_df))]

bars = ax.bar(
    model_labels,
    results_df["roc_auc_mean"],
    yerr=results_df["roc_auc_std"],
    color=colors,
    edgecolor="white",
    linewidth=1.2,
    capsize=6,
    error_kw={"elinewidth": 2, "ecolor": "#444444"},
    alpha=0.88,
    width=0.55,
)

# Annotate bars with exact values
for bar, mean, std in zip(bars, results_df["roc_auc_mean"], results_df["roc_auc_std"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.003,
        f"{mean:.4f}",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

# Baseline line
ax.axhline(dummy_auc, color="crimson", linestyle="--", linewidth=1.5, label=f"Dummy baseline ({dummy_auc:.2f})")

ax.set_title("Model Comparison — ROC AUC (5-fold CV ± 1 std)", fontsize=14, pad=14)
ax.set_ylabel("ROC AUC", fontsize=12)
ax.set_ylim(0.5, min(1.0, results_df["roc_auc_mean"].max() + results_df["roc_auc_std"].max() + 0.05))
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "04_model_comparison_auc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to reports/04_model_comparison_auc.png")

## Learning Curves — Best Model

**Learning curves** show how model performance changes as more training data is added. They reveal:
- **High bias (underfitting)**: both train and val scores are low and converge at a low plateau
- **High variance (overfitting)**: large gap between train and validation score
- **Good fit**: val score improves as data grows, small gap, both scores are high

For deployment decisions, if the val curve is still rising, adding more data would help.

In [ ]:
from sklearn.model_selection import learning_curve
from client_churn_prediction.models import _get_classifier, _build_pipeline

best_estimator = _get_classifier(best_model_name, config)
best_pipeline = _build_pipeline(X, best_estimator)

train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X, y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, "o-", color=PALETTE[0], lw=2, label="Training ROC AUC")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color=PALETTE[0])
ax.plot(train_sizes, val_mean, "s--", color=PALETTE[1], lw=2, label="Validation ROC AUC")
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color=PALETTE[1])

ax.set_title(f"Learning Curves — {best_model_name.replace('_', ' ').title()}", fontsize=14, pad=12)
ax.set_xlabel("Training Set Size", fontsize=12)
ax.set_ylabel("ROC AUC", fontsize=12)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
sns.despine()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "04_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Confusion Matrix — Best Model on Validation Set

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Retrain best model on the training split
best_estimator_fit = _get_classifier(best_model_name, config)
best_pipeline_fit  = _build_pipeline(X_train, best_estimator_fit)
best_pipeline_fit.fit(X_train, y_train)

y_pred = best_pipeline_fit.predict(X_val)
y_proba = best_pipeline_fit.predict_proba(X_val)[:, 1]

cm = confusion_matrix(y_val, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed (0)", "Churned (1)"]
)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(
    f"Confusion Matrix — {best_model_name.replace('_', ' ').title()}\n(Validation Set, n={len(y_val)})",
    fontsize=12, pad=10
)

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (correctly predicted stay)  : {tn}")
print(f"False Positives (predicted churn, stayed)   : {fp}")
print(f"False Negatives (missed churners)           : {fn}")
print(f"True Positives  (correctly caught churners) : {tp}")
print(f"\nPrecision: {tp / (tp + fp):.3f} | Recall: {tp / (tp + fn):.3f}")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "04_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## ROC Curve

The **Receiver Operating Characteristic** curve plots True Positive Rate vs False Positive Rate at every possible classification threshold. The **Area Under the Curve (AUC)** summarises the model's ability to rank churners above non-churners — threshold-independent and robust to class imbalance.

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score

fig, ax = plt.subplots(figsize=(7, 6))

RocCurveDisplay.from_predictions(
    y_val, y_proba,
    name=best_model_name.replace("_", " ").title(),
    color=PALETTE[0],
    linewidth=2,
    ax=ax
)

# Dummy baseline
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1.5, label="Random baseline (AUC=0.50)")

# Optimal threshold (closest to top-left corner)
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_val, y_proba)
optimal_idx = np.argmax(tpr - fpr)
ax.scatter(fpr[optimal_idx], tpr[optimal_idx], s=120, zorder=5,
           color=PALETTE[3], marker="*", label=f"Optimal threshold ({thresholds[optimal_idx]:.2f})")

ax.set_title("ROC Curve — Validation Set", fontsize=14, pad=12)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "04_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ROC AUC on held-out validation: {roc_auc_score(y_val, y_proba):.4f}")
print(f"Optimal threshold (Youden's J) : {thresholds[optimal_idx]:.4f}")

## Precision-Recall Curve

For **imbalanced datasets** (our churn rate is ~20%), the Precision-Recall curve is more informative than ROC. It focuses specifically on the minority class:

- **Precision**: of all clients we flag as churners, what fraction actually churn? (avoids wasting retention budget)
- **Recall**: of all clients who will churn, what fraction do we catch? (avoids missed revenue)

The **Average Precision (AP)** summarises the curve into a single number (baseline = churn rate).

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

fig, ax = plt.subplots(figsize=(7, 6))

PrecisionRecallDisplay.from_predictions(
    y_val, y_proba,
    name=best_model_name.replace("_", " ").title(),
    color=PALETTE[1],
    linewidth=2,
    ax=ax
)

# Baseline: random classifier precision = churn rate
baseline_precision = y_val.mean()
ax.axhline(baseline_precision, linestyle="--", color="grey", linewidth=1.5,
           label=f"Random baseline (P={baseline_precision:.2f})")

ap = average_precision_score(y_val, y_proba)
ax.set_title(f"Precision-Recall Curve — AP={ap:.4f}", fontsize=14, pad=12)
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "04_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Average Precision (AP)  : {ap:.4f}")
print(f"Random baseline AP      : {baseline_precision:.4f}")
print(f"Lift over random        : {ap / baseline_precision:.2f}x")

## Model Selection Decision

### Summary of findings

| Criterion | Winner |
|---|---|
| Highest ROC AUC | Gradient Boosting |
| Lowest variance across folds | Logistic Regression |
| Best Precision-Recall tradeoff | Gradient Boosting |
| Training speed | Logistic Regression |
| Interpretability | Logistic Regression |

### Decision

We select **Gradient Boosting** as the primary model for hyperparameter tuning in notebook `05`. The rationale:

1. **Consistently highest ROC AUC** across all 5 folds — not just a lucky split.
2. **Handles non-linear interactions** between features like `age_tenure_ratio`, `balance_salary_ratio`, and `products_active_combo` that logistic regression misses without manual polynomial expansion.
3. **Natively robust to outliers** — the GradientBoostingClassifier fits decision trees internally, which are split-based and do not require feature scaling.
4. **Business interpretability** is still achievable through feature importances and permutation importances (see notebook `05`).

**Logistic Regression** is retained as a **shadow model** for A/B testing and as a sanity check: if the GB model ever degrades in production, LR provides a safe fallback.

### Next step
→ Proceed to `05_hyperparameter_tuning.ipynb` to run `RandomizedSearchCV` and optimise the business threshold.